# 문맥 보존형 뉴스-KOSIS 기사 100건 평가

숫자가 포함된 기사 100건을 고정 seed로 선택해 전체 파이프라인을 실행하고 READY 도달률, 실제값 검증 성공률, 단계별 탈락 사유를 자동 집계합니다. 중단 후 같은 셀을 다시 실행하면 기존 진행 파일을 이어받습니다.

In [ ]:
from google.colab import drive, files, userdata
drive.mount('/content/drive')

import csv
import json
import os
import random
import shutil
import subprocess
import sys
from collections import Counter
from pathlib import Path

SAMPLE_SIZE = 100
SAMPLE_SEED = 20260730
REPO_URL = 'https://github.com/rnwjdgus03/NLP_05-Team-Project-3.git'
BRANCH = 'codex/repro-baseline-20260727'
REPO_DIR = Path('/content/NLP_05-Team-Project-3')
DRIVE_ROOT = Path('/content/drive/MyDrive/NLP_05-Team-Project-3')
INPUT_DIR = DRIVE_ROOT / 'inputs'
RUN_DIR = DRIVE_ROOT / 'runs' / f'contextual_eval_context_v2_{SAMPLE_SIZE}_seed{SAMPLE_SEED}'
INDEX_DIR = DRIVE_ROOT / 'indexes' / 'kosis_bge_m3'
ARTICLE_CSV = INPUT_DIR / 'news_articles.csv'
EVAL_ARTICLES = RUN_DIR / f'00_eval_articles_{SAMPLE_SIZE}.csv'
EARLY_META_CANDIDATES = [
    DRIVE_ROOT / 'runs' / 'early_bge_rag_5000' / 'early_bge_meta_index.csv',
    DRIVE_ROOT / 'runs' / 'early_bge_rag' / 'early_bge_meta_index.csv',
]
INPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('run:', RUN_DIR)

In [ ]:
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'requests>=2.31,<3', 'python-dotenv>=1.0,<2', 'kss>=6,<7',
    'numpy>=1.26,<3', 'sentence-transformers>=3.4,<6', 'transformers>=4.45,<6'
], check=True)
os.chdir(REPO_DIR)
print('commit:', subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip())

## 원문 기사 CSV

`MyDrive/NLP_05-Team-Project-3/inputs/news_articles.csv`가 없을 때만 업로드 창이 열립니다.

In [ ]:
if not ARTICLE_CSV.exists():
    print('기사 원문 CSV를 선택하세요.')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError('기사 원문 CSV 한 개만 업로드하세요.')
    shutil.copy2(next(iter(uploaded)), ARTICLE_CSV)
print('article CSV:', ARTICLE_CSV, ARTICLE_CSV.stat().st_size)

In [ ]:
from preprocess_news import read_articles, resolve_columns

if not EVAL_ARTICLES.exists():
    articles, fieldnames, encoding = read_articles(ARTICLE_CSV, 'auto')
    columns = resolve_columns(fieldnames, {})
    body_col = columns['body']
    numeric_articles = [row for row in articles if any(ch.isdigit() for ch in str(row.get(body_col, '') or ''))]
    if len(numeric_articles) < SAMPLE_SIZE:
        raise RuntimeError(f'숫자가 포함된 기사가 부족합니다: {len(numeric_articles)}')
    sample = random.Random(SAMPLE_SEED).sample(numeric_articles, SAMPLE_SIZE)
    with EVAL_ARTICLES.open('w', encoding='utf-8-sig', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(sample)
    print(f'articles={len(articles):,} numeric={len(numeric_articles):,} sample={SAMPLE_SIZE} encoding={encoding}')
else:
    with EVAL_ARTICLES.open(encoding='utf-8-sig', newline='') as handle:
        existing_count = sum(1 for _ in csv.DictReader(handle))
    if existing_count != SAMPLE_SIZE:
        raise RuntimeError(f'기존 표본 크기가 다릅니다: {existing_count} != {SAMPLE_SIZE}')
    print('고정 표본 재사용:', EVAL_ARTICLES)
print('evaluation input:', EVAL_ARTICLES)

In [ ]:
for key in ('CLOVA_API_KEY', 'KOSIS_API_KEY'):
    if not os.environ.get(key):
        os.environ[key] = userdata.get(key) or ''
    if not os.environ.get(key):
        raise RuntimeError(f'Colab 보안 비밀에 {key}를 등록하세요.')

for path, label in [
    (INDEX_DIR / 'manifest.json', 'BGE manifest'),
    (INDEX_DIR / 'embeddings.npy', 'BGE embeddings'),
    (REPO_DIR / 'kosis_table_summary.csv', 'KOSIS table index'),
]:
    if not path.exists():
        raise FileNotFoundError(f'{label}가 없습니다: {path}')
EARLY_META = next((path for path in EARLY_META_CANDIDATES if path.exists()), None)
print('inputs and secrets: ready')
print('early meta:', EARLY_META or '없음')

In [ ]:
command = [
    sys.executable, '-u', str(REPO_DIR / 'run_contextual_news_kosis_pipeline.py'),
    '--articles', str(EVAL_ARTICLES),
    '--table-index', str(REPO_DIR / 'kosis_table_summary.csv'),
    '--semantic-index', str(INDEX_DIR),
    '--out-dir', str(RUN_DIR),
    '--device', 'cuda',
    '--verify',
]
if EARLY_META:
    command.extend(['--early-meta-index', str(EARLY_META)])
print(' '.join(command))
subprocess.run(command, check=True)

In [ ]:
summary_path = RUN_DIR / '08_evaluation_summary.json'
if not summary_path.exists():
    subprocess.run([
        sys.executable, '-u', str(REPO_DIR / 'evaluate_contextual_kosis_run.py'),
        '--run-dir', str(RUN_DIR),
    ], check=True)

summary = json.loads(summary_path.read_text(encoding='utf-8'))
print('\n[주요 건수]')
for key, value in summary['counts'].items():
    print(f'{key:28s}: {value}')
print('\n[주요 비율]')
for key, value in summary['rates'].items():
    print(f'{key:28s}: {value}%')
print('\n[verdict]')
print(summary['verdicts'])
print('\n[mapping status]')
print(summary['mapping_statuses'])

In [ ]:
reasons_path = RUN_DIR / '08_evaluation_reasons.csv'
with reasons_path.open(encoding='utf-8-sig', newline='') as handle:
    reason_rows = list(csv.DictReader(handle))

print('[단계별 상위 사유]')
for stage in ('gate', 'mapping', 'verification'):
    print(f'\n{stage}')
    stage_rows = [row for row in reason_rows if row['stage'] == stage]
    stage_rows.sort(key=lambda row: int(row['count']), reverse=True)
    for row in stage_rows[:10]:
        print(row['bucket'], row['reason_code'], row['count'], row['reason_detail'][:120])

print('\n최종 파일')
for name in (
    '08_evaluation_summary.csv',
    '08_evaluation_reasons.csv',
    '08_evaluation_summary.json',
    '08_evaluation_report.md',
):
    print(RUN_DIR / name)